In [1]:
%pip install pandas openpyxl scikit-learn requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.cluster import DBSCAN
from math import radians, sin, cos, asin, sqrt
import requests
import time

# 충전소 위치 파일
location_path = Path(
    r"C:\test\08_지도시각화\서울시_충전소_위경도.xlsx"
)

# 충전소별 예측값 및 등급 파일
grade_path = Path(
    r"C:\test\06_코드및노트북\충전소별_대표예측충전량_구간분류.xlsx"
)

# 파일 확인
print("위치 파일 존재:", location_path.exists())
print("예측값 파일 존재:", grade_path.exists())

위치 파일 존재: True
예측값 파일 존재: True


In [10]:
# E등급(빨간색) 충전소만 추출
df_red = df[df["등급"].astype(str).str.strip() == "E"].copy()

print("전체 충전소 수:", len(df))
print("빨간색(E등급) 충전소 수:", len(df_red))

display(
    df_red[
        ["충전소명", "위도", "경도", "대표 예측값", "등급"]
    ].sort_values("대표 예측값", ascending=False)
)

전체 충전소 수: 181
빨간색(E등급) 충전소 수: 24


,충전소명,위도,경도,대표 예측값,등급
64,미아동 도봉세무서 앞 전기차 충전소,37.619753,127.028247,51.257055,E
134,으뜸공원 공영주차장 전기차 충전소,37.620429,127.026855,48.789280,E
66,미화어린이공원 주차장(가로등),37.655297,127.039732,38.912733,E
35,대궐재어린이공원 거주자주차장(가로등),37.574355,126.937118,36.656699,E
57,명일동 거주자주차장 134번(가로등),37.551761,127.154475,36.441721,E
69,방화로 노상공영주차장(가로등),37.554500,126.816435,35.779892,E
53,망원3 노상공영주차장(가로등),37.554912,126.900559,35.685183,E
155,창3동 제1공영주차장,37.633110,127.041197,35.437856,E
13,공릉동 거주자주차장(가로등),37.617654,127.090758,34.846845,E
12,곰달래5길 노상공영주차장(가로등),37.532035,126.833461,34.844547,E


In [19]:

from sklearn.cluster import DBSCAN
import pandas as pd
import numpy as np

results = []

for radius in [1000, 1500, 2000, 2500, 3000]:
    model = DBSCAN(
        eps=radius / 6_371_000,
        min_samples=2,
        metric="haversine"
    )

    labels = model.fit_predict(coords_red)

    cluster_count = len(set(labels) - {-1})
    clustered_count = int((labels != -1).sum())

    results.append({
        "반경(m)": radius,
        "군집 수": cluster_count,
        "군집 포함 충전소 수": clustered_count,
        "군집 미포함 충전소 수": int((labels == -1).sum()),
        "가장 큰 군집 크기": max(
            [int((labels == label).sum())
             for label in set(labels) if label != -1],
            default=0
        )
    })

display(pd.DataFrame(results))


,반경(m),군집 수,군집 포함 충전소 수,군집 미포함 충전소 수,가장 큰 군집 크기
0,1000,3,6,18,2
1,1500,6,12,12,2
2,2000,5,13,11,4
3,2500,5,16,8,7
4,3000,5,20,4,8


In [20]:

from sklearn.cluster import DBSCAN
import pandas as pd
import numpy as np

# 2,500m 기준으로 군집 생성
model = DBSCAN(
    eps=2500 / 6_371_000,
    min_samples=2,
    metric="haversine"
)

df_red = df_red.copy()
df_red["군집"] = model.fit_predict(
    np.radians(df_red[["위도", "경도"]].to_numpy())
)

# 군집에 포함된 충전소만 선택 (잡음 -1 제외)
df_clustered = df_red[df_red["군집"] != -1].copy()

# 군집별 중심 좌표 및 통계
summary = (
    df_clustered.groupby("군집")
    .agg(
        충전소수=("충전소명", "count"),
        평균예측충전량=("대표 예측값", "mean"),
        중심위도=("위도", "mean"),
        중심경도=("경도", "mean"),
        충전소목록=("충전소명", lambda x: ", ".join(x))
    )
    .reset_index()
    .sort_values("충전소수", ascending=False)
)

display(summary.round({
    "평균예측충전량": 2,
    "중심위도": 6,
    "중심경도": 6
}))

# 엑셀 저장
output_path = Path(r"C:\test\08_지도시각화\E등급_충전소_군집분석.xlsx")

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="군집요약", index=False)
    df_clustered.to_excel(writer, sheet_name="충전소별군집", index=False)

print("저장 완료:", output_path)
print("군집 포함 충전소:", len(df_clustered))
print("군집 미포함 충전소:", int((df_red["군집"] == -1).sum()))


,군집,충전소수,평균예측충전량,중심위도,중심경도,충전소목록
1,1,7,38.45,37.633745,127.041227,"노원세무서 거주자주차장(가로등), 미아동 도봉세무서 앞 전기차 충전소, 미화어린이공..."
0,0,3,33.05,37.525338,126.851066,"곰달래5길 노상공영주차장(가로등), 신월5동 복개 노상공영주차장(가로등), 양천구민..."
2,2,2,31.97,37.493360,127.038561,"논현 제2호주차장, 도곡2동 주민센터"
3,3,2,34.38,37.555299,126.907851,"망원3 노상공영주차장(가로등), 청기와2 노상공영주차장(가로등)"
4,4,2,34.99,37.551761,127.154475,"명일동 거주자주차장 127번(가로등), 명일동 거주자주차장 134번(가로등)"


저장 완료: C:\test\08_지도시각화\E등급_충전소_군집분석.xlsx
군집 포함 충전소: 16
군집 미포함 충전소: 8


In [21]:
import requests
import pandas as pd

# 본인이 발급받은 카카오 REST API 키 입력
KAKAO_REST_API_KEY = "d85ddb9c7adc1d683e375a4b625efbca"

headers = {
    "Authorization": f"KakaoAK {KAKAO_REST_API_KEY}"
}

def get_address(lat, lon):
    url = "https://dapi.kakao.com/v2/local/geo/coord2address.json"

    response = requests.get(
        url,
        headers=headers,
        params={"x": lon, "y": lat},
        timeout=10
    )

    response.raise_for_status()
    documents = response.json().get("documents", [])

    if not documents:
        return "주소 없음"

    item = documents[0]
    road = item.get("road_address")
    parcel = item.get("address")

    if road:
        return road.get("address_name", "주소 없음")
    if parcel:
        return parcel.get("address_name", "주소 없음")

    return "주소 없음"


summary["중심 주소"] = summary.apply(
    lambda row: get_address(row["중심위도"], row["중심경도"]),
    axis=1
)

display(
    summary[
        ["군집", "충전소수", "평균예측충전량",
         "중심위도", "중심경도", "중심 주소"]
    ].round(2)
)

summary.to_excel(
    r"C:\test\08_지도시각화\E등급_충전소_군집분석_주소포함.xlsx",
    index=False
)

print("주소 포함 결과 저장 완료")


,군집,충전소수,평균예측충전량,중심위도,중심경도,중심 주소
1,1,7,38.45,37.63,127.04,서울 도봉구 창동 502-33
0,0,3,33.05,37.53,126.85,서울 양천구 신정동 1087
2,2,2,31.97,37.49,127.04,서울 강남구 역삼동 795-10
3,3,2,34.38,37.56,126.91,서울 마포구 망원동 57-93
4,4,2,34.99,37.55,127.15,서울 강동구 명일동 51-1


주소 포함 결과 저장 완료
